In [1]:
import os
import csv
import time
import psutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from itertools import combinations
from sklearn.preprocessing import MinMaxScaler
import optuna
import logging
import sys
from functools import partial

# --- Reusable Functions ---
from ReusableFunctions.DataPreprocessing import DataPreprocessing
from ReusableFunctions.EvaluationMetrics import EvaluationMetrics as EM
from reproducibility_settings import set_global_seed
from ReusableFunctions.RecordBestModel import record_best_models


# ============================================================
# Reproducibility
# ============================================================
set_global_seed(seed=42, framework='torch')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")


# ============================================================
# Benchmarking Helper Functions
# ============================================================

def synchronize_device():
    """
    Synchronize CUDA operations so timing measurements are accurate.
    """
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def get_parameter_counts(model):
    """
    Returns:
        total_parameters
        trainable_parameters
    """
    total_parameters = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_parameters = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total_parameters, trainable_parameters


def estimate_lstm_macs_per_sample(model, sequence_length):
    """
    Approximate MACs required for one sample.

    For an LSTM layer:

        MACs per timestep
        ≈ 4 * H * (I + H)

    where:
        H = hidden size
        I = input size

    Factor 4 represents:
        input gate
        forget gate
        candidate/cell gate
        output gate

    Final fully-connected layer is also included.

    Bias operations, sigmoid, tanh, dropout, indexing and other
    element-wise operations are not included.

    Therefore this should be interpreted as an approximate MAC count.
    """

    lstm = model.lstm

    hidden_size = lstm.hidden_size
    num_layers = lstm.num_layers

    num_directions = (
        2 if lstm.bidirectional else 1
    )

    total_macs = 0

    for layer in range(num_layers):

        if layer == 0:
            layer_input_size = lstm.input_size
        else:
            layer_input_size = (
                hidden_size * num_directions
            )

        macs_per_timestep = (
            4
            * hidden_size
            * (layer_input_size + hidden_size)
        )

        layer_macs = (
            sequence_length
            * macs_per_timestep
            * num_directions
        )

        total_macs += layer_macs

    # Final fully connected layer
    fc_macs = (
        model.fc.in_features
        * model.fc.out_features
    )

    total_macs += fc_macs

    return int(total_macs)


def measure_inference_time(
    model,
    X,
    batch_size=256,
    warmup=True
):
    """
    Measure complete inference time over X.

    Returns:
        total_inference_time_seconds
        latency_per_sample_seconds
    """

    if X is None:
        return np.nan, np.nan

    if len(X) == 0:
        return np.nan, np.nan

    dataset = TensorDataset(
        torch.tensor(
            X,
            dtype=torch.float32
        )
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()

    # --------------------------------------------------------
    # Warm-up
    # --------------------------------------------------------
    if warmup:

        first_batch = next(iter(loader))[0].to(device)

        with torch.no_grad():

            for _ in range(3):
                _ = model(first_batch)

        synchronize_device()

    # --------------------------------------------------------
    # Actual timing
    # --------------------------------------------------------
    synchronize_device()

    inference_start = time.perf_counter()

    with torch.no_grad():

        for (xb,) in loader:

            xb = xb.to(device)

            _ = model(xb)

    synchronize_device()

    total_inference_time = (
        time.perf_counter()
        - inference_start
    )

    latency_per_sample = (
        total_inference_time
        / len(X)
    )

    return (
        total_inference_time,
        latency_per_sample
    )


# ============================================================
# Train & Evaluate
# ============================================================

def train_and_evaluate_model(
    model,
    train_loader,
    val_data,
    optimizer,
    criterion,
    scaler_y,
    forecast_window,
    epochs,
    patience=0.2
):

    best_val_loss = float('inf')

    epochs_no_improve = 0

    actual_patience = (
        int(patience * epochs)
        if isinstance(patience, float)
        else patience
    )

    # Prevent patience becoming 0
    actual_patience = max(
        1,
        actual_patience
    )

    X_val_tensor = torch.tensor(
        val_data[0],
        dtype=torch.float32
    ).to(device)

    y_val_tensor = torch.tensor(
        val_data[1],
        dtype=torch.float32
    ).to(device)

    # --------------------------------------------------------
    # Training timer
    # --------------------------------------------------------
    synchronize_device()

    training_start = time.perf_counter()

    epochs_completed = 0

    for epoch in range(epochs):

        # ====================================================
        # Training
        # ====================================================
        model.train()

        for xb, yb in train_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            predictions = model(xb).squeeze()

            loss = criterion(
                predictions,
                yb.squeeze()
            )

            loss.backward()

            optimizer.step()

        # ====================================================
        # Validation for early stopping
        # ====================================================
        model.eval()

        with torch.no_grad():

            val_preds_scaled = (
                model(X_val_tensor)
                .squeeze()
            )

            val_loss = criterion(
                val_preds_scaled,
                y_val_tensor.squeeze()
            ).item()

        epochs_completed = epoch + 1

        # ====================================================
        # Early stopping
        # ====================================================
        if val_loss < best_val_loss:

            best_val_loss = val_loss

            epochs_no_improve = 0

        else:

            epochs_no_improve += 1

            if (
                epochs_no_improve
                >= actual_patience
            ):
                break

    synchronize_device()

    training_time = (
        time.perf_counter()
        - training_start
    )

    # --------------------------------------------------------
    # Final validation prediction
    # --------------------------------------------------------
    model.eval()

    with torch.no_grad():

        val_preds_scaled = (
            model(X_val_tensor)
            .cpu()
            .numpy()
        )

    val_preds_unscaled = (
        scaler_y.inverse_transform(
            val_preds_scaled.reshape(-1, 1)
        )
    )

    y_val_true_unscaled = (
        scaler_y.inverse_transform(
            val_data[1].reshape(-1, 1)
        )
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    r2 = EM.r2(
        y_val_true_unscaled,
        val_preds_unscaled
    )

    if (
        np.isnan(r2)
        or np.isinf(r2)
    ):

        rmse = np.nan
        mae = np.nan
        acc = np.nan

    else:

        rmse = EM.rmse(
            y_val_true_unscaled,
            val_preds_unscaled
        )

        mae = EM.mae(y_val_true_unscaled, val_preds_unscaled)

        acc = EM.accuracy(
            y_val_true_unscaled,
            val_preds_unscaled
        )

    profit_index = (
        EM.profitability_index_v1(
            y_val_true_unscaled,
            val_preds_unscaled,
            forecast_window
        )
    )

    del X_val_tensor
    del y_val_tensor

    return (
        rmse,
        mae,
        r2,
        acc,
        profit_index,
        training_time,
        epochs_completed
    )


# ============================================================
# Optuna Logging
# ============================================================

optuna.logging.get_logger(
    "optuna"
).addHandler(
    logging.StreamHandler(
        sys.stdout
    )
)

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)


# ============================================================
# LSTM Model
# ============================================================

class LSTMModel(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        dropout,
        activation=None
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=(
                dropout
                if num_layers > 1
                else 0
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        out, _ = self.lstm(x)

        # Last timestep
        out = out[:, -1, :]

        out = self.dropout(out)

        return self.fc(out)


# ============================================================
# Optuna Objective
# ============================================================

def objective(
    trial,
    data,
    selected_indicators,
    ticker,
    window_size,
    forecast_window
):

    # ========================================================
    # Whole trial timer
    # ========================================================
    optimization_start = (
        time.perf_counter()
    )

    # ========================================================
    # CPU memory before
    # ========================================================
    process = psutil.Process(
        os.getpid()
    )

    cpu_memory_before = (
        process
        .memory_info()
        .rss
    )

    # ========================================================
    # Reset GPU peak memory
    # ========================================================
    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats(
            device
        )

    # ========================================================
    # Retrieve data
    # ========================================================
    X_train = data["X_train"]

    X_val = data["X_val"]

    y_train_scaled = data["y_train"]

    y_val_scaled = data["y_val"]

    scaler_y = data["scaler_y"]

    # --------------------------------------------------------
    # Test set is optional
    # --------------------------------------------------------
    X_test = data.get(
        "X_test",
        None
    )

    # ========================================================
    # Hyperparameters
    # ========================================================
    hyperparams = {

        "hidden_size":
            trial.suggest_categorical(
                "hidden_size",
                [32, 64, 128]
            ),

        #"dropout":
        #    trial.suggest_float(
        #        "dropout",
        #        0.2,
        #        0.8,
        #        step=0.1
        #    ),

        "dropout": trial.suggest_categorical(
                "dropout",
                [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
                ),

        "lr":
            trial.suggest_categorical(
                "lr",
                [
                    0.0001,
                    0.0005,
                    0.001
                ]
            ),

        "batch_size":
            trial.suggest_categorical(
                "batch_size",
                [16, 32, 64]
            ),

        "epochs":
            trial.suggest_categorical(
                "epochs",
                [50, 100, 150]
            )
    }

    # ========================================================
    # DataLoader
    # ========================================================
    train_loader = DataLoader(

        TensorDataset(

            torch.tensor(
                X_train,
                dtype=torch.float32
            ),

            torch.tensor(
                y_train_scaled,
                dtype=torch.float32
            )
        ),

        batch_size=(
            hyperparams[
                "batch_size"
            ]
        ),

        shuffle=False
    )

    # ========================================================
    # Create model
    # ========================================================
    model = LSTMModel(

        input_size=(
            X_train.shape[2]
        ),

        hidden_size=(
            hyperparams[
                "hidden_size"
            ]
        ),

        num_layers=1,

        dropout=(
            hyperparams[
                "dropout"
            ]
        )

    ).to(device)

    optimizer = optim.Adam(

        model.parameters(),

        lr=(
            hyperparams["lr"]
        )
    )

    criterion = nn.MSELoss()

    # ========================================================
    # Parameter counts
    # ========================================================
    (
        total_parameters,
        trainable_parameters
    ) = get_parameter_counts(
        model
    )

    # ========================================================
    # MACs / FLOPs
    # ========================================================
    sequence_length = (
        X_train.shape[1]
    )

    macs_per_sample = (
        estimate_lstm_macs_per_sample(
            model,
            sequence_length
        )
    )

    # Approximation:
    # 1 multiply + 1 addition = 2 FLOPs
    approx_flops_per_sample = (
        2 * macs_per_sample
    )

    # ========================================================
    # Train
    # ========================================================
    (
        rmse,
        mae,
        r2,
        acc,
        profit_index,
        training_time,
        epochs_completed

    ) = train_and_evaluate_model(

        model=model,

        train_loader=train_loader,

        val_data=(
            X_val,
            y_val_scaled
        ),

        optimizer=optimizer,

        criterion=criterion,

        scaler_y=scaler_y,

        forecast_window=(
            forecast_window
        ),

        epochs=(
            hyperparams["epochs"]
        )
    )

    # ========================================================
    # Inference benchmarks
    # ========================================================

    # Fixed inference batch size so comparisons are fair
    # across trials with different training batch sizes.
    inference_batch_size = 256

    # --------------------------------------------------------
    # Training inference time
    # --------------------------------------------------------
    (
        train_inference_time,
        train_latency_per_sample
    ) = measure_inference_time(

        model,

        X_train,

        batch_size=(
            inference_batch_size
        )
    )

    # --------------------------------------------------------
    # Validation inference time
    # --------------------------------------------------------
    (
        validation_inference_time,
        validation_latency_per_sample

    ) = measure_inference_time(

        model,

        X_val,

        batch_size=(
            inference_batch_size
        )
    )

    # --------------------------------------------------------
    # Test inference time
    # --------------------------------------------------------
    if (
        X_test is not None
        and len(X_test) > 0
    ):

        (
            test_inference_time,
            test_latency_per_sample

        ) = measure_inference_time(

            model,

            X_test,

            batch_size=(
                inference_batch_size
            )
        )

    else:

        test_inference_time = np.nan

        test_latency_per_sample = np.nan

    # ========================================================
    # GPU peak memory
    # ========================================================
    if torch.cuda.is_available():

        synchronize_device()

        peak_gpu_memory = (

            torch.cuda
            .max_memory_allocated(
                device
            )

            / (1024 ** 2)
        )

    else:

        peak_gpu_memory = np.nan

    # ========================================================
    # CPU memory
    # ========================================================
    cpu_memory_after = (
        process
        .memory_info()
        .rss
    )

    cpu_memory_change = (

        cpu_memory_after
        - cpu_memory_before

    ) / (1024 ** 2)

    # ========================================================
    # Complete Optuna trial time
    # ========================================================
    optimization_time = (

        time.perf_counter()
        - optimization_start
    )

    # ========================================================
    # Store Optuna attributes
    # Optional but very useful.
    # ========================================================

    trial.set_user_attr(
        "training_time",
        training_time
    )

    trial.set_user_attr(
        "optimization_time",
        optimization_time
    )

    trial.set_user_attr(
        "total_parameters",
        total_parameters
    )

    trial.set_user_attr(
        "trainable_parameters",
        trainable_parameters
    )

    trial.set_user_attr(
        "macs_per_sample",
        macs_per_sample
    )

    trial.set_user_attr(
        "flops_per_sample",
        approx_flops_per_sample
    )

    trial.set_user_attr(
        "peak_gpu_memory_mb",
        peak_gpu_memory
    )

    trial.set_user_attr(
        "cpu_memory_change_mb",
        cpu_memory_change
    )

    # ========================================================
    # Write results
    # ========================================================
    with open(

        f'stock_results/'
        f'{ticker}_LSTM_results.csv',

        'a',

        newline=''

    ) as f:

        writer = csv.writer(f)

        writer.writerow([

            # ------------------------------------------------
            # Trial / configuration
            # ------------------------------------------------
            trial.number,

            ', '.join(
                selected_indicators
            ),

            hyperparams[
                "hidden_size"
            ],

            round(hyperparams[
                "dropout"
            ],1),

            hyperparams[
                "lr"
            ],

            hyperparams[
                "batch_size"
            ],

            window_size,

            forecast_window,

            hyperparams[
                "epochs"
            ],

            epochs_completed,

            # ------------------------------------------------
            # Predictive performance
            # ------------------------------------------------
            rmse,
            mae,
            r2,

            acc,

            profit_index,

            # ------------------------------------------------
            # Timing
            # ------------------------------------------------
            round(training_time,6),

            round(optimization_time,6),

            round(train_inference_time,7),

            round(validation_inference_time,7),

            round(test_inference_time,7),

            test_latency_per_sample,

            # ------------------------------------------------
            # Model complexity
            # ------------------------------------------------
            total_parameters,

            trainable_parameters,

            macs_per_sample,

            approx_flops_per_sample,

            # ------------------------------------------------
            # Memory
            # ------------------------------------------------
            peak_gpu_memory,

            cpu_memory_change
        ])

    # ========================================================
    # Print trial benchmark
    # ========================================================



    # ========================================================
    # Cleanup
    # ========================================================
    del model
    del optimizer
    del train_loader

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    # ========================================================
    # Optuna objective
    # ========================================================
    if (
        np.isnan(rmse)
        or np.isinf(rmse)
    ):

        return 1e10

    return rmse


# ============================================================
# Main Execution
# ============================================================

if __name__ == "__main__":

    ticker = 'AAPL'

    # ========================================================
    # Results directory
    # ========================================================
    os.makedirs(
        'stock_results',
        exist_ok=True
    )

    # ========================================================
    # Result file
    # ========================================================
    result_file = (
        f'stock_results/'
        f'{ticker}_LSTM_results.csv'
    )

    # ========================================================
    # Create CSV header
    # ========================================================
    if not os.path.exists(
        result_file
    ):

        with open(
            result_file,
            'w',
            newline=''
        ) as f:

            writer = csv.writer(f)

            writer.writerow([

                # --------------------------------------------
                # Trial/configuration
                # --------------------------------------------
                "Trial",

                "Indicators",

                "Hidden Size",

                "Dropout",

                "LR",

                "Batch Size",

                "Window Size",

                "Forecast Window",

                "Requested Epochs",

                "Actual Epochs",

                # --------------------------------------------
                # Predictive metrics
                # --------------------------------------------
                "RMSE",
                "MAE",
                "R2",

                "Accuracy",

                "Profit Index",

                # --------------------------------------------
                # Time metrics
                # --------------------------------------------
                "Training Time (s)",

                "Optimization Time (s)",

                "Train Inference Time (s)",

                "Validation Inference Time (s)",

                "Test Inference Time (s)",

                "Test Latency / Sample (s)",

                # --------------------------------------------
                # Complexity
                # --------------------------------------------
                "Total Parameters",

                "Trainable Parameters",

                "MACs / Sample",

                "Approx FLOPs / Sample",

                # --------------------------------------------
                # Memory
                # --------------------------------------------
                "Peak GPU Memory (MB)",

                "CPU Memory Change (MB)"
            ])

    # ========================================================
    # Preprocessing
    # ========================================================
    data_processor = (
        DataPreprocessing(
            ticker=ticker
        )
    )

    df_with_all_indicators = (
        data_processor
        .add_technical_indicators()
    )

    # ========================================================
    # Save downloaded / processed data
    # ========================================================
    os.makedirs(
        'check',
        exist_ok=True
    )

    df_with_all_indicators.to_csv(

        f'check/'
        f'{ticker}_LSTM_downloaded_data.csv',

        index=True
    )

    # ========================================================
    # Indicators
    # ========================================================
    selected_base_indicators = [

        '20MA',

        '50MA',

        'RSI',

        'MACD',

        'Upper_BB',

        'Lower_BB',

        'CCI',

        'ATR',

        'Williams_%R',

        'OBV'
    ]

    # ========================================================
    # Select 6 indicators from 10
    # C(10,6) = 210 combinations
    # ========================================================
    all_combinations = list(

        combinations(
            selected_base_indicators,
            6
        )
    )

    # ========================================================
    # Window / forecast combinations
    # ========================================================
    window_forecast_combos = [

        (60, 1),

        (60, 30),

        (5, 1)
    ]

    # ========================================================
    # Resume settings
    # ========================================================

    start_combo_index = 0

    start_config_index = 0

    # --------------------------------------------------------
    # Resume calculation:
    #
    # Step 1:
    # Remove incomplete rows until the previous Optuna cycle
    # ends at Trial 24.
    #
    # Step 2:
    #
    # completed_configurations =
    #     (CSV rows - 1 header) / 25
    #
    # Each indicator combination has 3 window/forecast
    # configurations.
    #
    # start_combo_index =
    #     completed_configurations // 3
    #
    # start_config_index =
    #     completed_configurations % 3
    #
    # Examples:
    #
    # 15726 lines:
    #
    # (15726 - 1) / 25
    # = 629 completed configurations
    #
    # 629 // 3 = 209
    # 629 % 3  = 2
    #
    # Therefore:
    #
    # start_combo_index = 209
    # start_config_index = 2
    #
    #
    # 15701 lines:
    #
    # (15701 - 1) / 25
    # = 628
    #
    # 628 // 3 = 209
    # 628 % 3 = 1
    #
    # start_combo_index = 209
    # start_config_index = 1
    #
    #
    # 15676 lines:
    #
    # (15676 - 1) / 25
    # = 627
    #
    # 627 // 3 = 209
    # 627 % 3 = 0
    #
    # start_combo_index = 209
    # start_config_index = 0
    # --------------------------------------------------------

    # ========================================================
    # Indicator combinations
    # ========================================================
    for i, indicator_combo in enumerate(
        all_combinations
    ):

        if i < start_combo_index:
            continue

        # ====================================================
        # Window / forecast combinations
        # ====================================================
        for j, (
            window_size,
            forecast_window
        ) in enumerate(
            window_forecast_combos
        ):

            if (
                i == start_combo_index
                and
                j < start_config_index
            ):
                continue

            #print(
            #    f"\n"
            #    f"==============================================\n"
            #    f"[{i + 1}/"
            #    f"{len(all_combinations)}]\n"
            #    f"Combo: {indicator_combo}\n"
            #    f"Window: {window_size}\n"
            #    f"Forecast: {forecast_window}\n"
            #    f"=============================================="
            #)
            print(f"[j:{j}|cur:{i + 1}\n")

            # =================================================
            # Prepare data
            # =================================================
            data = (
                data_processor
                .prepare_data(

                    ['Close']
                    + list(
                        indicator_combo
                    ),

                    window_size=(
                        window_size
                    ),

                    forecast_window=(
                        forecast_window
                    )
                )
            )

            # =================================================
            # Check dataset shapes
            # =================================================
            print(
                "X_train shape:",
                data["X_train"].shape
            )

            print(
                "X_val shape:",
                data["X_val"].shape
            )

            if (
                "X_test" in data
                and
                data["X_test"] is not None
            ):

                print(
                    "X_test shape:",
                    data[
                        "X_test"
                    ].shape
                )

            else:

                print(
                    "X_test not available."
                )

            # =================================================
            # Optuna
            # =================================================
            study = (
                optuna.create_study(
                    direction='minimize'
                )
            )

            study.optimize(

                partial(

                    objective,

                    data=data,

                    selected_indicators=(
                        indicator_combo
                    ),

                    ticker=ticker,

                    window_size=(
                        window_size
                    ),

                    forecast_window=(
                        forecast_window
                    )
                ),

                n_trials=25
            )

            # =================================================
            # Print best trial for current configuration
            # =================================================
            print(
                "\nBest trial for this "
                "configuration:"
            )

            print(
                "Best RMSE:",
                study.best_value
            )

            print(
                "Best Parameters:",
                study.best_params
            )

    # ========================================================
    # Record overall best models
    # ========================================================
    record_best_models(
        result_file
    )

GPU Name: NVIDIA GeForce RTX 5080


[*********************100%***********************]  1 of 1 completed


Removed 61 rows corresponding to ex-dividend and earnings dates.
Null values in each column:
 Price        Ticker
Close        AAPL      0
High         AAPL      0
Low          AAPL      0
Open         AAPL      0
Volume       AAPL      0
20MA                   0
50MA                   0
RSI                    0
12EMA                  0
26EMA                  0
MACD                   0
Signal_Line            0
Upper_BB               0
Lower_BB               0
CCI                    0
ATR                    0
Williams_%R            0
OBV                    0
dtype: int64
Does the dataset contain any null values? False
[j:0|cur:1

X_train shape: (1852, 60, 7)
X_val shape: (264, 60, 7)
X_test not available.

Best trial for this configuration:
Best RMSE: 3.3427
Best Parameters: {'hidden_size': 64, 'dropout': 0.2, 'lr': 0.0005, 'batch_size': 16, 'epochs': 150}
[j:1|cur:1

X_train shape: (1831, 60, 7)
X_val shape: (261, 60, 7)
X_test not available.

Best trial for this configuration:
Best RM